# Protein pathogenicity analysis workflow

This jupyter notebook is for **pathogenicity analysis of missense mutations in different protein regions**. It was used to analyze GLUT protein family responsible for glucose/fructose. Regions of focus are extracllular, intracellular and transmembrane domains, lining residues of tunnels, binding pockets and parts of lining residues that do not belong to binding pockets.

In [2]:
import source.helpers
import biolib
from pathlib import Path
import pandas as pd

from source import analysis, data, predict, regions
from source import path_method as pm

In [3]:
# some libraries that might be neeeded
#pip install pymissense
#pip3 install --upgrade pybiolib 
#pip install gunicorn

Analyzed proteins. You can choose order of analysis to go in GLUT type groups by toggling `grouped` variable. It will change output heatmap image.

In [4]:
grouped = True
if grouped:
    proteins = {
    'GLUT1':'P11166',
    'GLUT2':'P11168',
    'GLUT3':'P11169',
    'GLUT4':'P14672',
    'GLUT14':'Q8TDB8',#group 1
    'GLUT5':'P22732',
    'GLUT7':'Q6PXP3',
    'GLUT9':'Q9NRM0',
    'GLUT11':'Q9BYW1',#group2
    'GLUT6':'Q9UGQ3',
    'GLUT8':'Q9NY64',
    'GLUT10':'O95528',
    'GLUT12':'Q8TD20',
    'GLUT13':'Q96QE2' #group3
}
else:
    proteins = {
    'GLUT1':'P11166',
    'GLUT2':'P11168',
    'GLUT3':'P11169',
    'GLUT4':'P14672',
    'GLUT5':'P22732',
    'GLUT6':'Q9UGQ3',
    'GLUT7':'Q6PXP3',
    'GLUT8':'Q9NY64',
    'GLUT9':'Q9NRM0',
    'GLUT10':'O95528',
    'GLUT11':'Q9BYW1',
    'GLUT12':'Q8TD20',
    'GLUT13':'Q96QE2',
    'GLUT14':'Q8TDB8' 
    }

## Get the data

download needed PDB and FASTA files

In [5]:
RES_DIR = Path.cwd() / 'unified_results'
DATA_DIR = Path.cwd() / 'unified_data'

#if results and data direcotries not exists create them
Path.mkdir(RES_DIR, exist_ok=True)
Path.mkdir(DATA_DIR, exist_ok=True)

In [6]:
#import pdb and fasta files
PDB_DIR = DATA_DIR / 'pdb'
FAS_DIR = DATA_DIR / 'fasta'

Path.mkdir(PDB_DIR, exist_ok=True)
Path.mkdir(FAS_DIR, exist_ok=True)
for up_id in proteins.values():
    
    data.get_pdb(up_id, PDB_DIR / f'{up_id}.pdb')
    data.get_fasta(up_id,FAS_DIR / f'{up_id}.fasta')

## Get predictions for proteins

**Pymissense** tool is used to predict pathogenicities for whole proteins. Predictions for **SIFT and PolyPhen-2 (Rhapsody) must be obtained manually**.

Expected directory for SIFT: `data/sift/`

Expected directory for PolyPhen-2: `data/rhapsody/`

**TIP**: if you have local copy of AlphaMissense TSV file use this. The file is 1.2 GB in ziped form and might take long time to download and extract.

In [7]:
alpha_missense_tsv = DATA_DIR / 'AlphaMissense_aa_substitutions.tsv'
data.get_alphamissense(alpha_missense_tsv)

AlphaMissense data already exists


In [9]:
# run pymissense for all proteins and generate csv of aminoacid pathogenicities

PYM_DIR = RES_DIR / 'pymissense'
Path.mkdir(PYM_DIR, exist_ok=True)

for up_id in proteins.values():
    pdb_file = PDB_DIR / f'{up_id}.pdb'
    
    print(f'running PyMissense for {up_id}')
    predict.pymissense(alpha_missense_tsv, pdb_file, up_id, PYM_DIR)

running PyMissense for P11166
AlphaMissense prediction for P11166 already exists
running PyMissense for P11168
AlphaMissense prediction for P11168 already exists
running PyMissense for P11169
AlphaMissense prediction for P11169 already exists
running PyMissense for P14672
AlphaMissense prediction for P14672 already exists
running PyMissense for Q8TDB8
AlphaMissense prediction for Q8TDB8 already exists
running PyMissense for P22732
AlphaMissense prediction for P22732 already exists
running PyMissense for Q6PXP3
AlphaMissense prediction for Q6PXP3 already exists
running PyMissense for Q9NRM0
AlphaMissense prediction for Q9NRM0 already exists
running PyMissense for Q9BYW1
AlphaMissense prediction for Q9BYW1 already exists
running PyMissense for Q9UGQ3
AlphaMissense prediction for Q9UGQ3 already exists
running PyMissense for Q9NY64
AlphaMissense prediction for Q9NY64 already exists
running PyMissense for O95528
AlphaMissense prediction for O95528 already exists
running PyMissense for Q8TD2

## Get protein regions

From PDB files we extract informations on positions of aminoacids and save it as csv with two columns for position (`residue_label`) and aminoacid type (`residue name`). Resulting files are saved to `results/regions/` directory.

In [10]:
#define dictionary of regions and their prefixes for file naming
reg_prefix = {'all':'all',
            'extracellular':'O',
            'membrane':'M',
            'intracellular':'I',
            'lining residues':'lr',
            'binding pocket':'bp'
            }

### Whole proteins

In [11]:
REG_DIR = RES_DIR / 'regions'
Path.mkdir(REG_DIR, exist_ok=True)

In [12]:
for up_id in proteins.values():
    regions.all_residues(pdb_file = DATA_DIR / f'pdb/{up_id}.pdb', out_file = REG_DIR/ f'{reg_prefix['all']}_{up_id}.csv')

### Intracellular, extracellular and membrane regions

These regions are assigned to aminoacids automatically using **[DeepTMHMM](https://dtu.biolib.com/DeepTMHMM)** prediction method.

In [14]:
#get annotations of AA positions relative to the membrane using 
DTM_DIR = RES_DIR / 'deeptmhmm'
Path.mkdir(DTM_DIR, exist_ok=True)

deeptmhmm = biolib.load('DTU/DeepTMHMM')
for up_id in proteins.values():
    predict.deepTMHMM(deeptmhmm, FAS_DIR / f'{up_id}.fasta', up_id, DTM_DIR)

2026-05-01 10:21:08,378 | INFO : Loaded project DTU/DeepTMHMM:1.0.57
DeepTMHMM prediction for P11166 already exists
DeepTMHMM prediction for P11168 already exists
DeepTMHMM prediction for P11169 already exists
DeepTMHMM prediction for P14672 already exists
DeepTMHMM prediction for Q8TDB8 already exists
DeepTMHMM prediction for P22732 already exists
DeepTMHMM prediction for Q6PXP3 already exists
DeepTMHMM prediction for Q9NRM0 already exists
DeepTMHMM prediction for Q9BYW1 already exists
DeepTMHMM prediction for Q9UGQ3 already exists
DeepTMHMM prediction for Q9NY64 already exists
DeepTMHMM prediction for O95528 already exists
DeepTMHMM prediction for Q8TD20 already exists
DeepTMHMM prediction for Q96QE2 already exists


In [15]:
#processing of depptmhmm results
intracel = reg_prefix['intracellular']
extracel = reg_prefix['extracellular']
memb = reg_prefix['membrane']
for up_id in proteins.values():
    regions.membrane_residues(DTM_DIR / f'{up_id}.3line', REG_DIR, up_id, extracel, intracel, memb)

Analysis focuses on the intramembrane regions so we need to identify extramembrane head and tail of the sequence. Useful information will be lenghts of original protein and extramembrane termini and position where extramembrane tail starts in an original sequence. 

In [28]:
def get_extramembrane_parts(DTHMM_file):
    with open(DTHMM_file, "r") as f:
        lines = f.read().splitlines()
    annotation = lines[2].strip()

    whole_lenght = len(annotation)
    head_found = False

    for i in range(1, whole_lenght):

        if not head_found and annotation[i] == 'M':
            head_found = True
            head_length = i
        if annotation[i] != 'M' and annotation[i-1] == 'M':
            tail_length = whole_lenght - i
            new_end = i
    return {'head': head_length, 'tail': tail_length, 'length': whole_lenght, 'new_end': new_end}

    

In [29]:
get_extramembrane_parts(DTM_DIR / 'O95528.3line')

{'head': 7, 'tail': 45, 'length': 541, 'new_end': 496}

541